# MedFlow — 00 · Ingestão de dados (camada Bronze)

Este notebook **somente ingere e preserva** as fontes do DATASUS, Ministério
da Saúde e IBGE. Não aplica regra de negócio, de/para, filtro analítico,
imputação nem cálculo de indicador. Os únicos acréscimos são colunas técnicas
de linhagem e um manifesto com contagens, esquema e hashes.

**Entradas:** SIH/RD, CNES/LT, API de localidades do IBGE, API DEMAS de
regiões e estabelecimentos, CONCLA/IBGE e referência CID-10 do DATASUS.  
**Saídas:** `dados/bronze/`, mantendo `dados/raw/` como cache dos arquivos DBC/DBF.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from ftplib import FTP
from hashlib import sha256
from pathlib import Path
from datetime import datetime, timezone
import gzip
import json
import subprocess
import urllib.request
from zipfile import ZipFile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import datasus_dbc
from dbfread import DBF

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_CACHE = BASE / "dados" / "raw"
DIR_BRONZE = BASE / "dados" / "bronze"
for pasta in (DIR_CACHE, DIR_BRONZE):
    pasta.mkdir(parents=True, exist_ok=True)

UF = "SP"
COMPETENCIAS = [(ano, mes) for ano in (2022, 2023) for mes in range(1, 13)]
FTP_HOST = "ftp.datasus.gov.br"
FTP_DIRS = {
    "RD": "/dissemin/publicos/SIHSUS/200801_/Dados",
    "LT": "/dissemin/publicos/CNES/200508_/Dados/LT",
}
SOBRESCREVER = False

def nome_arquivo(grupo, ano, mes):
    return f"{grupo}{UF}{str(ano)[2:]}{mes:02d}.dbc"

print("cache :", DIR_CACHE)
print("bronze:", DIR_BRONZE)
print("recorte:", UF, COMPETENCIAS[0], "a", COMPETENCIAS[-1])

## 1. Download das fontes

O download é idempotente: arquivos existentes no cache não são substituídos.
Um arquivo parcial só recebe o nome definitivo quando o download termina.

In [ ]:
def baixar_grupo(grupo):
    alvos = [nome_arquivo(grupo, ano, mes) for ano, mes in COMPETENCIAS]
    faltantes = [nome for nome in alvos if not (DIR_CACHE / nome).exists()]
    if not faltantes:
        print(f"[{grupo}] 24 arquivos no cache")
        return
    ftp = FTP(FTP_HOST, timeout=120)
    try:
        ftp.login()
        ftp.cwd(FTP_DIRS[grupo])
        remotos = {nome.upper(): nome for nome in ftp.nlst()}
        for nome in faltantes:
            real = remotos.get(nome.upper())
            assert real, f"arquivo ausente no FTP: {nome}"
            destino = DIR_CACHE / nome
            parcial = destino.with_suffix(".dbc.parcial")
            with parcial.open("wb") as arquivo:
                ftp.retrbinary(f"RETR {real}", arquivo.write)
            parcial.rename(destino)
            print("baixado:", nome)
    finally:
        try:
            ftp.quit()
        except Exception:
            ftp.close()

for grupo in ("RD", "LT"):
    baixar_grupo(grupo)

## 2. Descompressão DBC → DBF

O DBF é uma representação técnica do arquivo recebido. Nenhum registro ou
campo é alterado nesta etapa.

In [ ]:
for grupo in ("RD", "LT"):
    for ano, mes in COMPETENCIAS:
        dbc = DIR_CACHE / nome_arquivo(grupo, ano, mes)
        dbf = dbc.with_suffix(".dbf")
        if not dbf.exists():
            datasus_dbc.decompress(str(dbc), str(dbf))
            print("convertido:", dbf.name)

## 3. Serialização fiel em Parquet

Os valores entregues pelo leitor DBF são mantidos. `_arquivo_fonte`,
`_ano_arquivo` e `_mes_arquivo` são metadados de linhagem, derivados do nome
do arquivo — não são atributos clínicos.

In [ ]:
def consolidar(grupo, nome_saida):
    destino = DIR_BRONZE / nome_saida
    if destino.exists() and not SOBRESCREVER:
        print("já existe, validaremos sem substituir:", destino.name)
        return

    temporario = destino.with_suffix(".parquet.parcial")
    writer = None
    colunas_ref = None
    total = 0
    try:
        for ano, mes in COMPETENCIAS:
            dbf = (DIR_CACHE / nome_arquivo(grupo, ano, mes)).with_suffix(".dbf")
            frame = pd.DataFrame(iter(DBF(str(dbf), encoding="iso-8859-1")))
            frame["_arquivo_fonte"] = dbf.name
            frame["_ano_arquivo"] = ano
            frame["_mes_arquivo"] = mes
            if colunas_ref is None:
                colunas_ref = list(frame.columns)
            else:
                assert set(frame.columns) == set(colunas_ref), (
                    f"mudança de esquema em {dbf.name}: "
                    f"faltam={set(colunas_ref)-set(frame.columns)}, "
                    f"sobram={set(frame.columns)-set(colunas_ref)}"
                )
                frame = frame[colunas_ref]
            tabela = pa.Table.from_pandas(frame, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporario, tabela.schema, compression="snappy")
            writer.write_table(tabela.cast(writer.schema))
            total += len(frame)
            print(f"[{grupo}] {ano}-{mes:02d}: {len(frame):,} | acumulado {total:,}")
    finally:
        if writer is not None:
            writer.close()
    temporario.rename(destino)

consolidar("RD", "sih_rd_sp_2022_2023.parquet")
consolidar("LT", "cnes_lt_sp_2022_2023.parquet")

## 4. Referência bruta do IBGE

A resposta JSON é salva byte a byte. A transformação para dimensão municipal
pertence à Silver.

In [ ]:
URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios"
ARQ_IBGE = DIR_BRONZE / "ibge_municipios_sp_raw.json"
if not ARQ_IBGE.exists() or SOBRESCREVER:
    with urllib.request.urlopen(URL_IBGE, timeout=60) as resposta:
        conteudo_ibge = resposta.read()
    ARQ_IBGE.write_bytes(conteudo_ibge)
else:
    conteudo_ibge = ARQ_IBGE.read_bytes()
if conteudo_ibge.startswith(b"\x1f\x8b"):
    conteudo_ibge = gzip.decompress(conteudo_ibge)
    ARQ_IBGE.write_bytes(conteudo_ibge)
print("IBGE:", len(json.loads(conteudo_ibge)), "registros")

## 5. Referências cadastrais e terminológicas oficiais

As respostas são preservadas na Bronze. O cadastro de estabelecimentos é uma
fotografia **atual** consultada por CNES e será identificado como enriquecimento
não histórico na Silver. Região de saúde é obtida pela API oficial por
município; CID-10 vem do pacote oficial do DATASUS; natureza jurídica é
preservada a partir da página CONCLA/IBGE.

In [ ]:
def baixar_referencia(url, destino, timeout=180):
    if destino.exists() and not SOBRESCREVER:
        return destino.read_bytes()
    requisicao = urllib.request.Request(
        url, headers={"User-Agent": "MedFlow-FIAP/1.0", "Accept-Encoding": "identity"}
    )
    with urllib.request.urlopen(requisicao, timeout=timeout) as resposta:
        conteudo = resposta.read()
    if conteudo.startswith(b"\x1f\x8b"):
        conteudo = gzip.decompress(conteudo)
    destino.write_bytes(conteudo)
    return conteudo


URL_REGIOES = (
    "https://apidadosabertos.saude.gov.br/"
    "macrorregiao-e-regiao-de-saude/municipio?sigla_uf=SP&limit=860&offset=0"
)
ARQ_REGIOES = DIR_BRONZE / "ms_regioes_saude_sp_raw.json"
regioes_bytes = baixar_referencia(URL_REGIOES, ARQ_REGIOES)
regioes_payload = json.loads(regioes_bytes)
regioes = regioes_payload["macrorregiao_regiao_saude_municipios"]

URL_CID10 = "http://www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip"
ARQ_CID10 = DIR_BRONZE / "datasus_cid10_2008.zip"
# O servidor legado da CID-10 expira via urllib, mas responde via curl.
if not ARQ_CID10.exists() or SOBRESCREVER:
    subprocess.run([
        "curl", "-L", "--max-time", "120", "-sS", URL_CID10,
        "-o", str(ARQ_CID10),
    ], check=True)
with ZipFile(ARQ_CID10) as pacote:
    arquivos_cid = pacote.namelist()

URL_CONCLA = (
    "https://concla.ibge.gov.br/documentacao/3051-concla/estrutura/"
    "natureza-juridica-2021.html"
)
ARQ_CONCLA = DIR_BRONZE / "ibge_concla_natureza_juridica_2021.html"
baixar_referencia(URL_CONCLA, ARQ_CONCLA)

URL_CNES_MODELO = "https://apidadosabertos.saude.gov.br/cnes/estabelecimentos/{cnes}"
ARQ_CNES_ATUAL = DIR_BRONZE / "ms_cnes_estabelecimentos_atuais_raw.json"

if ARQ_CNES_ATUAL.exists() and not SOBRESCREVER:
    cnes_atual_payload = json.loads(ARQ_CNES_ATUAL.read_bytes())
else:
    codigos_cnes = sorted(
        pd.read_parquet(
            DIR_BRONZE / "sih_rd_sp_2022_2023.parquet", columns=["CNES"]
        ).CNES.astype("string").str.strip().unique()
    )

    def consultar_cnes(codigo):
        ultimo_erro = None
        for tentativa in range(3):
            try:
                requisicao = urllib.request.Request(
                    URL_CNES_MODELO.format(cnes=codigo),
                    headers={"User-Agent": "MedFlow-FIAP/1.0"},
                )
                with urllib.request.urlopen(requisicao, timeout=30) as resposta:
                    return codigo, json.loads(resposta.read())
            except Exception as erro:
                ultimo_erro = erro
        return codigo, {"_erro": str(ultimo_erro)}

    respostas = {}
    with ThreadPoolExecutor(max_workers=12) as executor:
        futuros = [executor.submit(consultar_cnes, codigo) for codigo in codigos_cnes]
        for futuro in as_completed(futuros):
            codigo, resposta = futuro.result()
            respostas[codigo] = resposta

    falhas = {codigo: item for codigo, item in respostas.items() if "_erro" in item}
    assert not falhas, f"falhas na API CNES: {list(falhas.items())[:10]}"
    cnes_atual_payload = {
        "fonte": URL_CNES_MODELO,
        "extraido_em_utc": datetime.now(timezone.utc).isoformat(),
        "observacao": "cadastro atual; usar somente como enriquecimento não histórico",
        "registros": [respostas[codigo] for codigo in codigos_cnes],
    }
    ARQ_CNES_ATUAL.write_text(
        json.dumps(cnes_atual_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )

print("regiões/municípios MS:", len(regioes))
print("arquivos no pacote CID-10:", len(arquivos_cid))
print("estabelecimentos CNES atuais:", len(cnes_atual_payload["registros"]))
print("CONCLA natureza jurídica:", ARQ_CONCLA.stat().st_size, "bytes")

## 6. Manifesto e validação da Bronze

Esta validação responde apenas se a ingestão está completa e reproduzível.
Validade semântica dos domínios é responsabilidade do notebook Silver.

In [ ]:
def hash_arquivo(caminho, bloco=1024 * 1024):
    digest = sha256()
    with caminho.open("rb") as arquivo:
        for parte in iter(lambda: arquivo.read(bloco), b""):
            digest.update(parte)
    return digest.hexdigest()

arquivos = {
    "sih": DIR_BRONZE / "sih_rd_sp_2022_2023.parquet",
    "cnes": DIR_BRONZE / "cnes_lt_sp_2022_2023.parquet",
    "ibge": ARQ_IBGE,
    "regioes_saude_ms": ARQ_REGIOES,
    "cid10_datasus": ARQ_CID10,
    "natureza_juridica_concla": ARQ_CONCLA,
    "cnes_estabelecimentos_atuais": ARQ_CNES_ATUAL,
}
sih_meta = pq.ParquetFile(arquivos["sih"])
cnes_meta = pq.ParquetFile(arquivos["cnes"])
ibge_qtd = len(json.loads(ARQ_IBGE.read_bytes()))

checks = {
    "arquivos_rd_cache": len(list(DIR_CACHE.glob("RDSP2[23][01][0-9].dbc"))),
    "arquivos_lt_cache": len(list(DIR_CACHE.glob("LTSP2[23][01][0-9].dbc"))),
    "linhas_sih": sih_meta.metadata.num_rows,
    "colunas_sih": len(sih_meta.schema_arrow.names),
    "linhas_cnes": cnes_meta.metadata.num_rows,
    "colunas_cnes": len(cnes_meta.schema_arrow.names),
    "municipios_ibge": ibge_qtd,
    "municipios_regiao_saude_ms": len(regioes),
    "estabelecimentos_cnes_atuais": len(cnes_atual_payload["registros"]),
    "arquivos_pacote_cid10": len(arquivos_cid),
}
esperado = {
    "arquivos_rd_cache": 24, "arquivos_lt_cache": 24,
    "linhas_sih": 5_210_357, "colunas_sih": 116,
    "linhas_cnes": 200_075, "colunas_cnes": 31,
    "municipios_ibge": 645,
    "municipios_regiao_saude_ms": 645,
    "estabelecimentos_cnes_atuais": 669,
    "arquivos_pacote_cid10": 6,
}
for chave, valor in checks.items():
    print(f"{chave:<24} {valor:>10,} | esperado {esperado[chave]:>10,}")
    assert valor == esperado[chave], f"{chave}: {valor} != {esperado[chave]}"

manifesto = {
    "camada": "bronze",
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
    "recorte": {"uf": UF, "competencias": [f"{a}{m:02d}" for a, m in COMPETENCIAS]},
    "principio": "fontes preservadas; sem regra de negócio, de/para ou filtro analítico",
    "fontes": {
        "SIH_RD": FTP_DIRS["RD"],
        "CNES_LT": FTP_DIRS["LT"],
        "IBGE_municipios": URL_IBGE,
        "MS_DEMAS_regioes_saude": URL_REGIOES,
        "MS_DEMAS_cnes_atual": URL_CNES_MODELO,
        "DATASUS_CID10": URL_CID10,
        "IBGE_CONCLA_natureza_juridica": URL_CONCLA,
    },
    "checks": checks,
    "arquivos": {
        nome: {
            "caminho": caminho.name,
            "bytes": caminho.stat().st_size,
            "sha256": hash_arquivo(caminho),
        }
        for nome, caminho in arquivos.items()
    },
}
(DIR_BRONZE / "MANIFESTO.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("\nBRONZE VÁLIDA — ingestão completa; seguir para o notebook 01.")